In [28]:
import os

print(os.getcwd())

/Users/lynn/Desktop/engg2112-soil-moisture-prediction-main/notebooks


In [29]:
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [30]:
# Load prepared data

data = np.load(
    "../data/processed/site_16023302_model_data.npz",
    allow_pickle=True
)

X_train = data["X_train"]
y_train = data["y_train"]

X_validation = data["X_validation"]
y_validation = data["y_validation"]

X_test = data["X_test"]
y_test = data["y_test"]

print("Training:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

Training: (296, 14, 1)
Validation: (64, 14, 1)
Test: (64, 14, 1)


In [31]:
# Reshape data for Random Forest

X_train_rf = X_train.reshape(X_train.shape[0], -1)

X_validation_rf = X_validation.reshape(
    X_validation.shape[0], -1
)

X_test_rf = X_test.reshape(X_test.shape[0], -1)

print("Training:", X_train_rf.shape)
print("Validation:", X_validation_rf.shape)
print("Test:", X_test_rf.shape)

Training: (296, 14)
Validation: (64, 14)
Test: (64, 14)


In [32]:
# Build Random Forest model

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

print(rf_model)

RandomForestRegressor(max_depth=5, random_state=42)


In [33]:
# Train Random Forest

rf_model.fit(X_train_rf, y_train)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [34]:
# Predict validation data

rf_val_predictions = rf_model.predict(X_validation_rf)

# Calculate validation metrics
val_mae = mean_absolute_error(
    y_validation,
    rf_val_predictions
)

val_rmse = np.sqrt(
    mean_squared_error(
        y_validation,
        rf_val_predictions
    )
)

print("Validation MAE:", val_mae)
print("Validation RMSE:", val_rmse)

Validation MAE: 0.0468961382839132
Validation RMSE: 0.08622641887696153


In [35]:
# Save trained Random Forest model

joblib.dump(
    rf_model,
    "../models/random_forest_soil_moisture.pkl"
)

print("Random Forest model saved successfully.")

Random Forest model saved successfully.


In [36]:
# Generate Random Forest predictions on test data

rf_test_predictions = rf_model.predict(X_test_rf)

print("Prediction shape:", rf_test_predictions.shape)

Prediction shape: (64,)


In [37]:
# Load fitted scaler

scaler = joblib.load(
    "../data/processed/site_16023302_minmax_scaler.pkl"
    
)

# Inverse transform predictions
rf_predictions_original = scaler.inverse_transform(
    rf_test_predictions.reshape(-1, 1)
).flatten()

# Inverse transform actual values
y_test_original = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).flatten()

print("Predicted ASM:", rf_predictions_original[:5])
print("Actual ASM:", y_test_original[:5])

Predicted ASM: [0.19823069 0.18284639 0.2005891  0.22831975 0.19925155]
Actual ASM: [0.189 0.203 0.235 0.21  0.204]


/Users/lynn/Desktop/engg2112-soil-moisture-prediction-main/.venv/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [38]:
# Save Random Forest test predictions

test_results_rf = pd.DataFrame({
    "Date": pd.to_datetime(data["dates_test"]),
    "Actual_ASM": y_test_original,
    "Random_Forest_Prediction": rf_predictions_original
})

test_results_rf.to_csv(
    "../results/random_forest_test_predictions.csv",
    index=False
)

print("Random Forest test predictions saved successfully.")

display(test_results_rf.head())

Random Forest test predictions saved successfully.


,Date,Actual_ASM,Random_Forest_Prediction
0,2018-04-24,0.189,0.198231
1,2018-04-25,0.203,0.182846
2,2018-04-26,0.235,0.200589
3,2018-04-27,0.210,0.228320
4,2018-04-28,0.204,0.199252


## Random Forest Model Configuration

### Model
- Model: Random Forest Regressor
- Input: Previous 14 days of ASM
- Output: Next-day ASM
- Number of trees: 100
- Maximum tree depth: 5
- Random state: 42

### Dataset
- Training samples: 296
- Validation samples: 64
- Test samples: 64
- Normalisation: MinMaxScaler

### Training
- Training method: Random Forest Regression
- No epochs or batch size required
- No gradient-based optimiser required
- No early stopping required

### Evaluation
- Validation metrics: MAE and RMSE
- Test metrics: MAE and RMSE
- Comparison models: BiLSTM and Persistence Baseline